# TP 06 — Siamese networks, BPR, and the course leaderboard

**Recommender Systems — SCIA — Session 06 (last session)**

Last session's two-tower network was trained to say *yes or no* for one (user, item) pair at a time. A recommender has to **rank**. Today you train for ranking directly: a **siamese network** that scores a movie the user liked and a movie they did not with the *same* item tower, and is trained so that the liked one comes out on top (Bayesian Personalised Ranking). Then you put every model of the course on one leaderboard and decide, with evidence, which one you would ship.

**Required standards:** modular code, numpy-style docstring on every function, self-contained notebook.


In [ ]:
import io
import os
import itertools
import zipfile
import urllib.request

os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from typing import Iterable, Sequence
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RELEVANCE_THRESHOLD = 4.0   # a held-out rating >= 4.0 means "the user liked it"
K = 10                      # length of the recommendation lists


## 1. Data and split

Same MovieLens `ml-latest-small` and temporal split as Sessions 04 and 05.

In [ ]:
def download_movielens(dest_dir: str, url: str = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip") -> str:
    """Download and extract the MovieLens ml-latest-small archive.

    Parameters
    ----------
    dest_dir : str
        Directory in which the archive is extracted. Created if needed.
    url : str, optional
        URL of the MovieLens zip archive.

    Returns
    -------
    str
        Path to the extracted ``ml-latest-small`` folder. If it already
        exists, nothing is downloaded.
    """
    os.makedirs(dest_dir, exist_ok=True)
    extracted_path = os.path.join(dest_dir, "ml-latest-small")
    if os.path.isdir(extracted_path):
        return extracted_path
    with urllib.request.urlopen(url) as response:
        archive_bytes = response.read()
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        archive.extractall(dest_dir)
    return extracted_path


def load_movielens(data_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load the MovieLens ratings and movies tables.

    Parameters
    ----------
    data_dir : str
        Path to the ``ml-latest-small`` folder.

    Returns
    -------
    ratings : pandas.DataFrame
        Columns ``userId``, ``movieId``, ``rating``, ``timestamp``.
    movies : pandas.DataFrame
        Columns ``movieId``, ``title``, ``genres``.
    """
    ratings = pd.read_csv(os.path.join(data_dir, "ratings.csv"))
    movies = pd.read_csv(os.path.join(data_dir, "movies.csv"))
    return ratings, movies


data_dir = download_movielens(dest_dir="./data")
ratings, movies = load_movielens(data_dir)
print(f"{len(ratings):,} ratings, {ratings['userId'].nunique()} users, {ratings['movieId'].nunique():,} rated movies")


Provided: tools from the earlier sessions, unchanged. The split and metrics (Session 04), the beyond-accuracy metrics (Session 04), the feature functions and the two-tower network (Session 05), and the genre and latent-factor models (Sessions 02 and 03). They are the competitors on today's leaderboard.

In [ ]:
def temporal_user_split(ratings: pd.DataFrame, test_frac: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Per-user temporal split: each user's most recent ratings go to test.

    Parameters
    ----------
    ratings : pandas.DataFrame
        Columns ``userId``, ``movieId``, ``rating``, ``timestamp``.
    test_frac : float, optional
        Fraction of each user's ratings to hold out (at least one).

    Returns
    -------
    train, test : pandas.DataFrame
        Index reset. Ties in ``timestamp`` keep their file order.
    """
    ordered = ratings.sort_values(["userId", "timestamp"], kind="mergesort")
    position = ordered.groupby("userId").cumcount()
    n_ratings = ordered.groupby("userId")["rating"].transform("size")
    n_test = np.maximum(1, np.floor(n_ratings * test_frac).astype(int))
    is_test = position >= (n_ratings - n_test)
    return ordered[~is_test].reset_index(drop=True), ordered[is_test].reset_index(drop=True)


def restrict_to_known_items(train: pd.DataFrame, test: pd.DataFrame) -> pd.DataFrame:
    """Drop test ratings of items that never appear in the training set.

    Parameters
    ----------
    train, test : pandas.DataFrame
        Training and held-out ratings.

    Returns
    -------
    pandas.DataFrame
        The rows of ``test`` whose ``movieId`` occurs in ``train``.
    """
    return test[test["movieId"].isin(set(train["movieId"]))].reset_index(drop=True)


def precision_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Fraction of the first ``k`` recommended items that are relevant.

    Parameters
    ----------
    recommended : list of int
        Recommended item ids, best first.
    relevant : set of int
        Relevant item ids of the user.
    k : int
        List length.

    Returns
    -------
    float
        Precision@k.
    """
    return sum(1 for item in list(recommended)[:k] if item in relevant) / k


def recall_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Fraction of the relevant items found in the first ``k`` recommended items.

    Parameters
    ----------
    recommended : list of int
        Recommended item ids, best first.
    relevant : set of int
        Relevant item ids of the user.
    k : int
        List length.

    Returns
    -------
    float
        Recall@k (0.0 if ``relevant`` is empty).
    """
    if not relevant:
        return 0.0
    return sum(1 for item in list(recommended)[:k] if item in relevant) / len(relevant)


def ndcg_at_k(recommended: list[int], relevant: set[int], k: int) -> float:
    """Binary-relevance NDCG@k.

    Parameters
    ----------
    recommended : list of int
        Recommended item ids, best first.
    relevant : set of int
        Relevant item ids of the user.
    k : int
        List length.

    Returns
    -------
    float
        NDCG@k (0.0 if ``relevant`` is empty).
    """
    if not relevant:
        return 0.0
    top = list(recommended)[:k]
    gains = np.array([1.0 if item in relevant else 0.0 for item in top])
    dcg = float(np.sum(gains / np.log2(np.arange(2, len(top) + 2))))
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, min(len(relevant), k) + 2))))
    return dcg / idcg


def evaluate_recommender(recommender, test: pd.DataFrame, k: int = K, users=None,
                         threshold: float = RELEVANCE_THRESHOLD) -> pd.DataFrame:
    """Per-user precision, recall and NDCG of a recommender on held-out ratings.

    Only users with at least one relevant held-out item (rating >=
    ``threshold``) are evaluated.

    Parameters
    ----------
    recommender : ScoreRecommender
        Fitted recommender exposing ``recommend(user_id, k)``.
    test : pandas.DataFrame
        Held-out ratings.
    k : int, optional
        List length.
    users : iterable of int, optional
        Restrict the evaluation to these users.
    threshold : float, optional
        Minimum test rating for an item to be relevant.

    Returns
    -------
    pandas.DataFrame
        Indexed by ``userId``; columns ``precision``, ``recall``, ``ndcg``.
    """
    relevant_by_user = test[test["rating"] >= threshold].groupby("userId")["movieId"].apply(set).to_dict()
    wanted = sorted(relevant_by_user) if users is None else sorted(set(relevant_by_user) & set(users))
    rows = {}
    for user_id in wanted:
        recs = recommender.recommend(user_id, k)
        rel = relevant_by_user[user_id]
        rows[user_id] = {"precision": precision_at_k(recs, rel, k), "recall": recall_at_k(recs, rel, k),
                         "ndcg": ndcg_at_k(recs, rel, k)}
    per_user = pd.DataFrame.from_dict(rows, orient="index")
    per_user.index.name = "userId"
    return per_user


def summarise(per_user: pd.DataFrame) -> dict:
    """Mean metrics over users, with the standard error of the mean precision.

    Parameters
    ----------
    per_user : pandas.DataFrame
        Output of ``evaluate_recommender``.

    Returns
    -------
    dict
        ``precision``, ``precision_se`` (std / sqrt(n)), ``recall``, ``ndcg``, ``n_users``.
    """
    n = len(per_user)
    return {"precision": per_user["precision"].mean(),
            "precision_se": per_user["precision"].std(ddof=1) / np.sqrt(n),
            "recall": per_user["recall"].mean(), "ndcg": per_user["ndcg"].mean(), "n_users": n}


In [ ]:
class ScoreRecommender:
    """Base class: a recommender defined by one score per catalogue item.

    Subclasses implement ``fit`` and ``scores``; ``recommend`` is shared. All
    components fitted on the same training set share the same catalogue
    ``items_`` (sorted item ids), so their score vectors can be combined.

    Attributes
    ----------
    items_ : numpy.ndarray
        Catalogue: the items seen in training, sorted.
    item_pos_ : dict of {int: int}
        Item id to position in ``items_``.
    counts_ : numpy.ndarray
        Number of training ratings of each item.
    seen_ : dict of {int: set of int}
        Items each user rated in training.
    n_hist_ : dict of {int: int}
        Number of training ratings of each user.
    """

    def _prepare(self, train: pd.DataFrame) -> None:
        """Compute the catalogue and the per-user histories.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        """
        self.items_ = np.array(sorted(train["movieId"].unique()))
        self.item_pos_ = {int(m): i for i, m in enumerate(self.items_)}
        self.counts_ = train.groupby("movieId").size().reindex(self.items_).to_numpy(float)
        self.seen_ = train.groupby("userId")["movieId"].apply(set).to_dict()
        self.n_hist_ = train.groupby("userId").size().to_dict()

    def scores(self, user_id: int) -> np.ndarray:
        """Score every catalogue item for one user (implemented by subclasses).

        Parameters
        ----------
        user_id : int
            User id.

        Returns
        -------
        numpy.ndarray, shape (len(items_),)
            One score per item of ``items_``; higher is better.
        """
        raise NotImplementedError

    def recommend(self, user_id: int, k: int) -> list[int]:
        """Return the ``k`` best-scoring items the user has not rated in training.

        Parameters
        ----------
        user_id : int
            User id.
        k : int
            List length.

        Returns
        -------
        list of int
            Item ids, best first (ties broken by item order).
        """
        s = np.array(self.scores(user_id), dtype=float)
        for item in self.seen_.get(user_id, ()):
            j = self.item_pos_.get(item)
            if j is not None:
                s[j] = -np.inf
        return [int(self.items_[j]) for j in np.argsort(-s, kind="mergesort")[:k]]


class PopularityScorer(ScoreRecommender):
    """Score items by their number of training ratings (same list for everyone)."""

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "PopularityScorer":
        """Count the ratings of each item.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata (unused).

        Returns
        -------
        PopularityScorer
            The fitted component.
        """
        self._prepare(train)
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Popularity of every item, in [0, 1].

        Parameters
        ----------
        user_id : int
            User id (unused: the scores do not depend on the user).

        Returns
        -------
        numpy.ndarray
            ``counts_ / counts_.max()``.
        """
        return self.counts_ / self.counts_.max()




In [ ]:
def catalog_coverage(rec_lists: Iterable[Sequence[int]], n_catalogue: int) -> float:
    """Catalogue coverage of a set of recommendation lists.

    Parameters
    ----------
    rec_lists : iterable of sequence of int
        One recommendation list per user.
    n_catalogue : int
        Number of items in the catalogue.

    Returns
    -------
    float
        Number of distinct items appearing in at least one list, divided by
        ``n_catalogue``.
    """
    distinct = {item for recs in rec_lists for item in recs}
    return len(distinct) / n_catalogue


def intra_list_diversity(recommended: Sequence[int], item_vectors, item_index: dict[int, int]) -> float:
    """Intra-list diversity of one recommendation list.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids.
    item_vectors : scipy.sparse matrix or numpy.ndarray, shape (n_items, n_features)
        One feature vector per row.
    item_index : dict of {int: int}
        Maps an item id to its row in ``item_vectors``. Items missing from
        this mapping are ignored.

    Returns
    -------
    float
        Mean, over all pairs of items in the list, of
        ``1 - cosine_similarity``. Returns 0.0 if fewer than two items are
        known.
    """
    rows = [item_index[item] for item in recommended if item in item_index]
    if len(rows) < 2:
        return 0.0
    similarity = cosine_similarity(item_vectors[rows])
    upper = np.triu_indices(len(rows), k=1)
    return float(np.mean(1.0 - similarity[upper]))


def novelty(recommended: Sequence[int], item_counts: dict[int, int], n_users: int) -> float:
    """Novelty of one recommendation list, in bits.

    Parameters
    ----------
    recommended : sequence of int
        Recommended item ids.
    item_counts : dict of {int: int}
        Number of training users who rated each item. Items missing from the
        dictionary, or with a count below 1, are treated as having count 1.
    n_users : int
        Number of users in the training set.

    Returns
    -------
    float
        Mean over the list of ``-log2(count_i / n_users)``.
    """
    counts = np.maximum(np.array([item_counts.get(item, 0) for item in recommended], dtype=float), 1.0)
    return float(np.mean(-np.log2(counts / n_users)))


In [ ]:
def build_genre_vectors(movies: pd.DataFrame, items: np.ndarray):
    """TF-IDF vectors of the genres of each item.

    Parameters
    ----------
    movies : pandas.DataFrame
        Columns ``movieId`` and ``genres`` (pipe-separated).
    items : numpy.ndarray
        Item ids to vectorise, in the row order of the result.

    Returns
    -------
    vectors : scipy.sparse.csr_matrix
        One TF-IDF row per item.
    index : dict of {int: int}
        Item id to row of ``vectors``.
    """
    genres = movies.set_index("movieId").loc[items, "genres"].replace("(no genres listed)", "")
    vectoriser = TfidfVectorizer(tokenizer=lambda text: [g for g in text.split("|") if g],
                                 token_pattern=None, lowercase=False)
    return vectoriser.fit_transform(genres), {int(m): i for i, m in enumerate(items)}




class GenreContentScorer(ScoreRecommender):
    """Content-based component on genre TF-IDF vectors (Session 02 idea).

    The user profile is the mean vector of the items the user liked in
    training; items are scored by cosine similarity to it, plus a tiny
    popularity term that only breaks ties between items with identical genres.

    Parameters
    ----------
    threshold : float, optional
        Minimum training rating for an item to count as liked.
    pop_weight : float, optional
        Weight of the popularity tie-breaker.
    """

    def __init__(self, threshold: float = RELEVANCE_THRESHOLD, pop_weight: float = 0.001):
        self.threshold = threshold
        self.pop_weight = pop_weight

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "GenreContentScorer":
        """Build item vectors and per-user liked lists.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata with a ``genres`` column.

        Returns
        -------
        GenreContentScorer
            The fitted component.
        """
        self._prepare(train)
        self.vectors_, self.index_ = build_genre_vectors(movies, self.items_)
        liked = train[train["rating"] >= self.threshold]
        self.liked_ = liked.groupby("userId")["movieId"].apply(list).to_dict()
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Cosine similarity of every item to the user profile.

        Parameters
        ----------
        user_id : int
            User id.

        Returns
        -------
        numpy.ndarray
            Similarity in [0, 1] plus the popularity tie-breaker.
        """
        liked = self.liked_.get(user_id) or list(self.seen_.get(user_id, []))
        rows = [self.index_[item] for item in liked if item in self.index_]
        profile = np.asarray(self.vectors_[rows].mean(axis=0))
        return cosine_similarity(profile, self.vectors_).ravel() + self.pop_weight * self.counts_ / self.counts_.max()


class LatentFactorScorer(ScoreRecommender):
    """The Session 03 model (latent factors X, W, b) in plain NumPy.

    Predicted rating: ``X[i] . W[u] + b[u] + item_mean[i]``. The parameters
    minimise the Session 03 cost (squared error over observed ratings only,
    plus L2 regularisation) with Adam. Item means are shrunk towards the
    global mean, so an item with a single 5-star rating is not trusted.

    Parameters
    ----------
    n_features : int, optional
        Number of latent features.
    lambda_ : float, optional
        L2 regularisation strength.
    iterations : int, optional
        Number of full-batch Adam steps.
    learning_rate : float, optional
        Adam learning rate.
    shrink : float, optional
        Pseudo-count of the global mean when computing item means.
    seed : int, optional
        Seed of the parameter initialisation.

    Attributes
    ----------
    X_, W_, b_ : numpy.ndarray
        Learned item factors, user factors and user biases.
    mean_ : numpy.ndarray
        Shrunk item means, shape (n_items, 1).
    history_ : list of tuple
        ``(iteration, cost)`` pairs recorded during training.
    """

    def __init__(self, n_features: int = 30, lambda_: float = 10.0, iterations: int = 150,
                 learning_rate: float = 0.05, shrink: float = 10.0, seed: int = 0):
        self.n_features, self.lambda_, self.iterations = n_features, lambda_, iterations
        self.learning_rate, self.shrink, self.seed = learning_rate, shrink, seed

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "LatentFactorScorer":
        """Learn X, W and b on the training ratings.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata (unused).

        Returns
        -------
        LatentFactorScorer
            The fitted component.
        """
        self._prepare(train)
        users = np.array(sorted(train["userId"].unique()))
        self.user_pos_ = {int(u): j for j, u in enumerate(users)}
        n_items, n_users = len(self.items_), len(users)
        Y, R = np.zeros((n_items, n_users)), np.zeros((n_items, n_users))
        rows = train["movieId"].map(self.item_pos_).to_numpy()
        cols = train["userId"].map(self.user_pos_).to_numpy()
        Y[rows, cols] = train["rating"].to_numpy()
        R[rows, cols] = 1
        g = train["rating"].mean()
        self.mean_ = ((Y.sum(1) + self.shrink * g) / (R.sum(1) + self.shrink)).reshape(-1, 1)
        Y_norm = (Y - self.mean_) * R

        rng = np.random.default_rng(self.seed)
        X = rng.normal(0, 0.1, (n_items, self.n_features))
        W = rng.normal(0, 0.1, (n_users, self.n_features))
        b = np.zeros((1, n_users))
        params = [X, W, b]
        m = [np.zeros_like(p) for p in params]
        v = [np.zeros_like(p) for p in params]
        self.history_ = []
        for t in range(1, self.iterations + 1):
            error = (X @ W.T + b - Y_norm) * R
            grads = (error @ W + self.lambda_ * X, error.T @ X + self.lambda_ * W, error.sum(0, keepdims=True))
            for i, (p, grad) in enumerate(zip(params, grads)):
                m[i] = 0.9 * m[i] + 0.1 * grad
                v[i] = 0.999 * v[i] + 0.001 * grad ** 2
                p -= self.learning_rate * (m[i] / (1 - 0.9 ** t)) / (np.sqrt(v[i] / (1 - 0.999 ** t)) + 1e-8)
            if t == 1 or t % 25 == 0:
                cost = 0.5 * float((error ** 2).sum()) + 0.5 * self.lambda_ * float((X ** 2).sum() + (W ** 2).sum())
                self.history_.append((t, cost))
        self.X_, self.W_, self.b_ = X, W, b
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Predicted rating of every item for one user.

        Parameters
        ----------
        user_id : int
            User id. A user absent from training gets the item means.

        Returns
        -------
        numpy.ndarray
            Predicted ratings (roughly on the 1-5 scale).
        """
        j = self.user_pos_.get(user_id)
        if j is None:
            return self.mean_.ravel().copy()
        return self.X_ @ self.W_[j] + self.b_[0, j] + self.mean_.ravel()


In [ ]:
def genre_list(movies: pd.DataFrame) -> list[str]:
    """Sorted list of the genres that appear in a movies table.

    Parameters
    ----------
    movies : pandas.DataFrame
        Columns ``movieId`` and ``genres`` (pipe-separated).

    Returns
    -------
    list of str
        Every genre found, alphabetically, without ``"(no genres listed)"``.
    """
    found = {g for text in movies["genres"] for g in text.split("|")}
    return sorted(found - {"(no genres listed)"})


In [ ]:
def build_item_features(train: pd.DataFrame, movies: pd.DataFrame, shrink: float = 10.0) -> tuple[np.ndarray, np.ndarray]:
    """Feature matrix of the catalogue, computed from the training set only.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings (``userId``, ``movieId``, ``rating``, ``timestamp``).
    movies : pandas.DataFrame
        Columns ``movieId``, ``title`` (ending with the release year in
        parentheses, e.g. ``"Heat (1995)"``) and ``genres``.
    shrink : float, optional
        Pseudo-count of the global mean rating used for the shrunk item mean.

    Returns
    -------
    items : numpy.ndarray, shape (n_items,)
        Sorted ids of the items that appear in ``train``.
    features : numpy.ndarray, shape (n_items, n_genres + 3)
        One row per item, columns in this order: the multi-hot encoding of
        the genres (order of ``genre_list(movies)``); the release year (items
        whose title has no year get the median year); the shrunk mean rating
        ``(sum + shrink * global_mean) / (count + shrink)``; and
        ``log1p(number of training ratings)``. ``global_mean`` is the mean
        of all training ratings.
    """
    genres = genre_list(movies)
    items = np.array(sorted(train["movieId"].unique()))
    meta = movies.set_index("movieId").loc[items]
    multi_hot = np.array([[1.0 if g in text.split("|") else 0.0 for g in genres] for text in meta["genres"]])
    year = meta["title"].str.extract(r"\((\d{4})\)\s*$")[0].astype(float)
    year = year.fillna(year.median()).to_numpy()
    stats = train.groupby("movieId")["rating"].agg(["sum", "size"]).reindex(items)
    shrunk = ((stats["sum"] + shrink * train["rating"].mean()) / (stats["size"] + shrink)).to_numpy()
    return items, np.column_stack([multi_hot, year, shrunk, np.log1p(stats["size"].to_numpy())])


def build_user_features(train: pd.DataFrame, movies: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Taste profile of every user, computed from the training set only.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings (``userId``, ``movieId``, ``rating``, ``timestamp``).
    movies : pandas.DataFrame
        Columns ``movieId`` and ``genres`` (pipe-separated).

    Returns
    -------
    users : numpy.ndarray, shape (n_users,)
        Sorted ids of the users that appear in ``train``.
    features : numpy.ndarray, shape (n_users, n_genres + 2)
        One row per user, columns in this order: for each genre (order of
        ``genre_list(movies)``) the mean rating the user gave to movies of
        that genre, or 0.0 if the user rated none; ``log1p(number of
        training ratings of the user)``; and the user's mean rating.
    """
    genres = genre_list(movies)
    users = np.array(sorted(train["userId"].unique()))
    genres_of = movies.set_index("movieId")["genres"].str.split("|")
    exploded = train.assign(genre=train["movieId"].map(genres_of)).explode("genre")
    per_genre = (exploded.pivot_table(index="userId", columns="genre", values="rating", aggfunc="mean")
                 .reindex(index=users, columns=genres).fillna(0.0))
    counts = train.groupby("userId").size().reindex(users)
    means = train.groupby("userId")["rating"].mean().reindex(users)
    return users, np.column_stack([per_genre.to_numpy(), np.log1p(counts.to_numpy()), means.to_numpy()])


In [ ]:
def build_two_tower(
    n_user_features: int,
    n_item_features: int,
    embedding_dim: int = 32,
    hidden: tuple[int, ...] = (128, 64),
    output: str = "probability",
):
    """Build a two-tower model: one network for users, one for items.

    Each tower is an MLP with ReLU hidden layers of the sizes in ``hidden``,
    followed by a linear layer of ``embedding_dim`` units and an L2
    normalisation, so that every embedding has unit norm. The score of a
    (user, item) pair is built from the cosine similarity of the two
    embeddings.

    Parameters
    ----------
    n_user_features : int
        Number of user features.
    n_item_features : int
        Number of item features.
    embedding_dim : int, optional
        Size of the embeddings.
    hidden : tuple of int, optional
        Sizes of the hidden layers of each tower.
    output : {"probability", "cosine"}, optional
        ``"cosine"``: the score is the cosine similarity itself, in [-1, 1].
        ``"probability"``: the score is a sigmoid of an affine function of the
        cosine (with learnable slope and bias), in (0, 1).

    Returns
    -------
    model : keras.Model
        Takes the list ``[user_features, item_features]`` (two inputs, in this
        order) and returns one score per pair, shape ``(batch, 1)``.
    user_tower : keras.Model
        Maps user features to unit-norm embeddings. Shares its weights with
        ``model``.
    item_tower : keras.Model
        Maps item features to unit-norm embeddings. Shares its weights with
        ``model``.

    Notes
    -----
    Keras 3: do not apply TensorFlow functions (``tf.*``) to symbolic Keras
    tensors.
    """
    def tower(n_in: int, name: str) -> keras.Model:
        """One tower: MLP, linear embedding layer, L2 normalisation.

        Parameters
        ----------
        n_in : int
            Number of input features.
        name : str
            Name prefix of the tower.

        Returns
        -------
        keras.Model
            Features to unit-norm embedding.
        """
        inputs = keras.Input(shape=(n_in,), name=f"{name}_features")
        x = inputs
        for units in hidden:
            x = keras.layers.Dense(units, activation="relu")(x)
        x = keras.layers.Dense(embedding_dim)(x)
        x = keras.layers.UnitNormalization(axis=-1)(x)
        return keras.Model(inputs, x, name=f"{name}_tower")

    user_tower, item_tower = tower(n_user_features, "user"), tower(n_item_features, "item")
    user_in = keras.Input(shape=(n_user_features,), name="user_features")
    item_in = keras.Input(shape=(n_item_features,), name="item_features")
    cosine = keras.layers.Dot(axes=1)([user_tower(user_in), item_tower(item_in)])
    score = cosine if output == "cosine" else keras.layers.Dense(1, activation="sigmoid")(cosine)
    return keras.Model([user_in, item_in], score, name="two_tower"), user_tower, item_tower


In [ ]:
def sample_training_pairs(
    train: pd.DataFrame,
    user_ids: np.ndarray,
    item_ids: np.ndarray,
    n_neg: int = 4,
    threshold: float = RELEVANCE_THRESHOLD,
    seed: int = 0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Build (user, item, label) training pairs for a retrieval-style model.

    Every training rating ``>= threshold`` gives one **positive** pair
    (label 1.0). For every positive pair, ``n_neg`` **negative** pairs
    (label 0.0) are added with the same user and an item drawn uniformly at
    random from the catalogue among the items the user has **not** rated in
    training.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings (``userId``, ``movieId``, ``rating``, ``timestamp``).
    user_ids : numpy.ndarray
        Sorted user ids; position ``j`` is the row of user ``user_ids[j]`` in
        the user feature matrix.
    item_ids : numpy.ndarray
        Sorted item ids (the catalogue); position ``i`` is the row of item
        ``item_ids[i]`` in the item feature matrix.
    n_neg : int, optional
        Negatives per positive.
    threshold : float, optional
        Minimum rating of a positive.
    seed : int, optional
        Seed of the random number generator.

    Returns
    -------
    user_rows : numpy.ndarray of int, shape (n_pos * (1 + n_neg),)
        Row of the user in the user feature matrix, for each pair.
    item_rows : numpy.ndarray of int, same shape
        Row of the item in the item feature matrix, for each pair.
    labels : numpy.ndarray of float, same shape
        1.0 for positives, 0.0 for negatives. All positives come first, in the
        order of their ratings in ``train``; the negatives follow, grouped by
        the positive they belong to.
    """
    rng = np.random.default_rng(seed)
    user_row = {int(u): j for j, u in enumerate(user_ids)}
    item_row = {int(m): i for i, m in enumerate(item_ids)}
    liked = train[train["rating"] >= threshold]
    pos_users = liked["userId"].map(user_row).to_numpy()
    pos_items = liked["movieId"].map(item_row).to_numpy()
    rated = [set() for _ in user_ids]
    for u, m in zip(train["userId"].map(user_row), train["movieId"].map(item_row)):
        rated[u].add(m)
    neg_users = np.repeat(pos_users, n_neg)
    neg_items = rng.integers(0, len(item_ids), size=len(neg_users))
    for k in range(len(neg_items)):
        while neg_items[k] in rated[neg_users[k]]:
            neg_items[k] = rng.integers(0, len(item_ids))
    user_rows = np.concatenate([pos_users, neg_users])
    item_rows = np.concatenate([pos_items, neg_items])
    labels = np.concatenate([np.ones(len(pos_users)), np.zeros(len(neg_users))])
    return user_rows, item_rows, labels


In [ ]:
class TwoTowerRecommender(ScoreRecommender):
    """Neural recommender: user and item features go through two towers, scores are dot products of embeddings.

    It is a *feature-combination* hybrid: the user tower sees a taste profile
    derived from ratings (a collaborative signal), the item tower sees content
    features. It uses the functions you wrote above.

    Parameters
    ----------
    objective : {"retrieval", "rating"}, optional
        ``"retrieval"``: positives and sampled negatives, binary cross-entropy.
        ``"rating"``: every training rating, target rescaled to [-1, 1], mean
        squared error (the objective of the original notebook).
    embedding_dim : int, optional
        Size of the embeddings.
    hidden : tuple of int, optional
        Hidden layer sizes of each tower.
    epochs : int, optional
        Training epochs.
    n_neg : int, optional
        Negatives per positive (retrieval only).
    batch_size : int, optional
        Mini-batch size.
    learning_rate : float, optional
        Adam learning rate.
    drop_popularity_features : bool, optional
        Zero the shrunk mean rating and log count columns of the item
        features, to see what the model does with content alone.
    seed : int, optional
        Seed for the weights initialisation and the negative sampling.

    Attributes
    ----------
    model_, user_tower_, item_tower_ : keras.Model
        The trained networks.
    user_emb_, item_emb_ : numpy.ndarray
        Embeddings of every user and of every catalogue item.
    history_ : list of float
        Training loss per epoch.
    """

    def __init__(self, objective: str = "retrieval", embedding_dim: int = 32, hidden: tuple = (128, 64),
                 epochs: int = 30, n_neg: int = 4, batch_size: int = 512, learning_rate: float = 1e-3,
                 drop_popularity_features: bool = False, seed: int = 1):
        self.objective, self.embedding_dim, self.hidden = objective, embedding_dim, hidden
        self.epochs, self.n_neg, self.batch_size = epochs, n_neg, batch_size
        self.learning_rate, self.drop_popularity_features, self.seed = learning_rate, drop_popularity_features, seed

    def _standardise_item(self, raw: np.ndarray) -> np.ndarray:
        """Standardise item features with the training mean and spread.

        Parameters
        ----------
        raw : numpy.ndarray
            Raw item feature rows.

        Returns
        -------
        numpy.ndarray
            Standardised rows.
        """
        return (raw - self.item_mu_) / self.item_sd_

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "TwoTowerRecommender":
        """Build the features, train the network and store the embeddings.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata.

        Returns
        -------
        TwoTowerRecommender
            The fitted recommender.
        """
        self._prepare(train)
        self.genres_ = genre_list(movies)
        self.global_mean_ = float(train["rating"].mean())
        item_ids, item_raw = build_item_features(train, movies)
        user_ids, user_raw = build_user_features(train, movies)
        assert np.array_equal(item_ids, self.items_), "item features must follow the sorted catalogue"
        if self.drop_popularity_features:
            item_raw[:, -2:] = 0.0
        self.item_mu_, self.item_sd_ = item_raw.mean(0), item_raw.std(0) + 1e-9
        self.user_mu_, self.user_sd_ = user_raw.mean(0), user_raw.std(0) + 1e-9
        item_x = self._standardise_item(item_raw)
        user_x = (user_raw - self.user_mu_) / self.user_sd_
        self.user_pos_ = {int(u): j for j, u in enumerate(user_ids)}

        keras.utils.set_random_seed(self.seed)
        if self.objective == "retrieval":
            u_rows, i_rows, y = sample_training_pairs(train, user_ids, item_ids, self.n_neg, seed=self.seed)
            output, loss = "probability", "binary_crossentropy"
        else:
            u_rows = train["userId"].map(self.user_pos_).to_numpy()
            i_rows = train["movieId"].map(self.item_pos_).to_numpy()
            y = (train["rating"].to_numpy() - 0.5) / 4.5 * 2 - 1
            output, loss = "cosine", "mse"
        self.model_, self.user_tower_, self.item_tower_ = build_two_tower(
            user_x.shape[1], item_x.shape[1], self.embedding_dim, self.hidden, output)
        self.model_.compile(optimizer=keras.optimizers.Adam(self.learning_rate), loss=loss)
        history = self.model_.fit([user_x[u_rows], item_x[i_rows]], y, epochs=self.epochs,
                                  batch_size=self.batch_size, shuffle=True, verbose=0)
        self.history_ = history.history["loss"]
        self.user_emb_ = self.user_tower_.predict(user_x, verbose=0)
        self.item_emb_ = self.item_tower_.predict(item_x, verbose=0)
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Cosine similarity between the user embedding and every item embedding.

        Parameters
        ----------
        user_id : int
            User id. A user unseen in training gets all-zero scores.

        Returns
        -------
        numpy.ndarray, shape (len(items_),)
            One score per catalogue item.
        """
        j = self.user_pos_.get(user_id)
        return np.zeros(len(self.items_)) if j is None else self.item_emb_ @ self.user_emb_[j]

    def predict_ratings(self, pairs: pd.DataFrame) -> np.ndarray:
        """Predicted rating of (user, item) pairs (meaningful for ``objective="rating"``).

        Parameters
        ----------
        pairs : pandas.DataFrame
            Columns ``userId`` and ``movieId``; users and items known from training.

        Returns
        -------
        numpy.ndarray
            Ratings on the 0.5 to 5 scale.
        """
        u = pairs["userId"].map(self.user_pos_).to_numpy()
        i = pairs["movieId"].map(self.item_pos_).to_numpy()
        cosine = np.sum(self.user_emb_[u] * self.item_emb_[i], axis=1)
        return np.clip((cosine + 1) / 2 * 4.5 + 0.5, 0.5, 5.0)

    def embed_new_item(self, genres: list[str], year: int) -> np.ndarray:
        """Embed a movie that has no ratings at all, from its content only.

        Parameters
        ----------
        genres : list of str
            Genres of the new movie.
        year : int
            Release year.

        Returns
        -------
        numpy.ndarray, shape (embedding_dim,)
            Unit-norm embedding. The movie is given the global mean rating and
            zero ratings count, as any unrated item would be.
        """
        multi_hot = [1.0 if g in genres else 0.0 for g in self.genres_]
        raw = np.array([multi_hot + [float(year), self.global_mean_, 0.0]])
        if self.drop_popularity_features:
            raw[:, -2:] = 0.0
        return self.item_tower_.predict(self._standardise_item(raw), verbose=0)[0]


def nearest_to_vector(vector: np.ndarray, item_embeddings: np.ndarray, item_ids: np.ndarray, k: int = 5) -> list[int]:
    """Ids of the items whose embeddings are closest (cosine) to an arbitrary vector.

    Parameters
    ----------
    vector : numpy.ndarray, shape (d,)
        Query embedding.
    item_embeddings : numpy.ndarray, shape (n_items, d)
        Catalogue embeddings.
    item_ids : numpy.ndarray
        Item id of each row.
    k : int, optional
        Number of items to return.

    Returns
    -------
    list of int
        Item ids, most similar first.
    """
    similarity = cosine_similarity(vector.reshape(1, -1), item_embeddings).ravel()
    return [int(item_ids[j]) for j in np.argsort(-similarity, kind="mergesort")[:k]]


def titles(ids: list[int], movies: pd.DataFrame) -> list[str]:
    """Titles of a list of movie ids.

    Parameters
    ----------
    ids : list of int
        Movie ids.
    movies : pandas.DataFrame
        Columns ``movieId`` and ``title``.

    Returns
    -------
    list of str
        One title per id.
    """
    return [movies.set_index("movieId").loc[i, "title"] for i in ids]


In [ ]:
def evaluate_full(recommender, test: pd.DataFrame, item_vectors, item_index: dict, item_counts: dict,
                  n_train_users: int, k: int = K, threshold: float = RELEVANCE_THRESHOLD):
    """Per-user accuracy and beyond-accuracy metrics of a recommender (Session 04 toolkit).

    Parameters
    ----------
    recommender : ScoreRecommender
        Fitted recommender.
    test : pandas.DataFrame
        Held-out ratings.
    item_vectors : scipy.sparse matrix
        Item feature vectors, for the intra-list diversity.
    item_index : dict of {int: int}
        Item id to row of ``item_vectors``.
    item_counts : dict of {int: int}
        Training rating counts per item, for the novelty.
    n_train_users : int
        Number of training users.
    k : int, optional
        List length.
    threshold : float, optional
        Minimum test rating for an item to be relevant.

    Returns
    -------
    per_user : pandas.DataFrame
        Indexed by ``userId``: ``precision``, ``recall``, ``ndcg``,
        ``diversity``, ``novelty``.
    rec_lists : dict of {int: list of int}
        The recommended list of every evaluated user.
    """
    relevant_by_user = test[test["rating"] >= threshold].groupby("userId")["movieId"].apply(set).to_dict()
    rows, rec_lists = {}, {}
    for user_id in sorted(relevant_by_user):
        recs = recommender.recommend(user_id, k)
        rel = relevant_by_user[user_id]
        rec_lists[user_id] = recs
        rows[user_id] = {"precision": precision_at_k(recs, rel, k), "recall": recall_at_k(recs, rel, k),
                         "ndcg": ndcg_at_k(recs, rel, k),
                         "diversity": intra_list_diversity(recs, item_vectors, item_index),
                         "novelty": novelty(recs, item_counts, n_train_users)}
    per_user = pd.DataFrame.from_dict(rows, orient="index")
    per_user.index.name = "userId"
    return per_user, rec_lists


def summarise_full(per_user: pd.DataFrame, rec_lists: dict, n_catalogue: int) -> dict:
    """One row of the leaderboard from the per-user metrics.

    Parameters
    ----------
    per_user : pandas.DataFrame
        Output of ``evaluate_full``.
    rec_lists : dict of {int: list of int}
        Output of ``evaluate_full``.
    n_catalogue : int
        Number of items in the catalogue.

    Returns
    -------
    dict
        ``precision``, ``precision_se``, ``ndcg``, ``coverage``,
        ``diversity``, ``novelty``.
    """
    n = len(per_user)
    return {"precision": per_user["precision"].mean(), "precision_se": per_user["precision"].std(ddof=1) / np.sqrt(n),
            "ndcg": per_user["ndcg"].mean(), "coverage": catalog_coverage(rec_lists.values(), n_catalogue),
            "diversity": per_user["diversity"].mean(), "novelty": per_user["novelty"].mean()}


In [ ]:
train, test_all = temporal_user_split(ratings, test_frac=0.2)
test = restrict_to_known_items(train, test_all)
print(f"train {len(train):,} rows | test {len(test):,} rows")


## 2. Exercise 1 — triplets: a liked movie against one that was not picked

A pairwise model learns from **comparisons**. For every movie a user liked, you need a movie they did not pick, to compare it with. Implement `sample_triplets`. It will be called at every epoch.

In [ ]:
def sample_triplets(
    train: pd.DataFrame,
    user_ids: np.ndarray,
    item_ids: np.ndarray,
    rng: np.random.Generator,
    threshold: float = RELEVANCE_THRESHOLD,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Draw one (user, positive item, negative item) triplet per positive rating.

    Every training rating ``>= threshold`` is a **positive** (user, item) pair.
    For each of them, one **negative** item is drawn uniformly at random from
    the catalogue among the items the user has **not** rated in training. The
    draw uses ``rng``, so that calling the function again gives fresh
    negatives.

    Parameters
    ----------
    train : pandas.DataFrame
        Training ratings (``userId``, ``movieId``, ``rating``, ``timestamp``).
    user_ids : numpy.ndarray
        Sorted user ids; position ``j`` is the row of user ``user_ids[j]`` in
        the user feature matrix.
    item_ids : numpy.ndarray
        Sorted item ids (the catalogue); position ``i`` is the row of item
        ``item_ids[i]`` in the item feature matrix.
    rng : numpy.random.Generator
        Source of randomness.
    threshold : float, optional
        Minimum rating of a positive.

    Returns
    -------
    user_rows, pos_rows, neg_rows : numpy.ndarray of int
        Three arrays of the same length (the number of positives): the rows
        of the user, of the positive item and of the negative item. Triplets
        follow the order of the positive ratings in ``train``.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
movies_t = pd.DataFrame({"movieId": [1, 2, 3], "title": ["A (1990)", "B (2000)", "C (2010)"],
                         "genres": ["Action|Comedy", "Comedy", "Drama"]})
train_t = pd.DataFrame({"userId": [10, 10, 20, 20], "movieId": [1, 2, 2, 3],
                        "rating": [4.0, 2.0, 5.0, 3.0], "timestamp": [1, 2, 3, 4]})

u_rows, p_rows, n_rows = sample_triplets(train_t, np.array([10, 20]), np.array([1, 2, 3]), np.random.default_rng(0))
print("triplets (user row, positive row, negative row):", list(zip(u_rows.tolist(), p_rows.tolist(), n_rows.tolist())), "(expected [(0, 0, 2), (1, 1, 0)])")
a = sample_triplets(train, np.sort(train["userId"].unique()), np.sort(train["movieId"].unique()), np.random.default_rng(1))
b = sample_triplets(train, np.sort(train["userId"].unique()), np.sort(train["movieId"].unique()), np.random.default_rng(2))
print("same positives, different negatives with another generator:", bool((a[1] == b[1]).all() and not (a[2] == b[2]).all()))


## 3. Exercise 2 — the BPR loss

For a user $u$, a liked item $i^+$ and a non-picked item $j$, the model outputs the score difference $x = s(u,i^+) - s(u,j)$. BPR maximises the probability $\sigma(x)$ that the liked item is ranked above, i.e. it minimises $-\log \sigma(x)$, averaged over the batch. Implement `bpr_loss`.

In [ ]:
def bpr_loss(y_true, y_pred):
    """Bayesian Personalised Ranking loss on a batch of score differences.

    The network outputs ``y_pred = s(u, i+) - s(u, i-)``, how much more it
    likes the positive item than the negative one. The loss is the mean of
    ``-log(sigmoid(y_pred))``: small when the positive item is ranked
    clearly above the negative one, large when it is ranked below.

    Parameters
    ----------
    y_true : keras tensor
        Ignored (Keras requires it; the targets carry no information).
    y_pred : keras tensor, shape (batch, 1)
        Score differences.

    Returns
    -------
    keras tensor
        Scalar loss. It must stay finite for score differences of any size
        (for instance +/- 1000).

    Notes
    -----
    Use ``keras.ops``: no ``tf.*`` functions.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
check = lambda x: float(bpr_loss(None, keras.ops.convert_to_tensor(np.array(x, dtype="float32").reshape(-1, 1))))
print("diff 0   :", round(check([0.0]), 4), "(expected 0.6931)")
print("diff +2  :", round(check([2.0]), 4), "(expected 0.1269)")
print("diff -2  :", round(check([-2.0]), 4), "(expected 2.1269)")
print("diff +-1000 finite:", bool(np.isfinite(check([1000.0, -1000.0]))), "(expected True)")


## 4. Exercise 3 — the siamese network

Build the model described in the docstring. The point to get right is the **weight sharing**: the item tower that scores the liked movie is *the same object* that scores the non-picked one, and the same goes for the item bias. You get back the full model for training and the pieces that you will later use alone to compute embeddings.

In [ ]:
def build_siamese_bpr(
    n_user_features: int,
    n_item_features: int,
    embedding_dim: int = 32,
    hidden: tuple[int, ...] = (128, 64),
    item_bias: bool = True,
):
    """Build a siamese network for pairwise ranking.

    One **user tower** and one **item tower**, each an MLP with ReLU hidden
    layers of the sizes in ``hidden`` followed by a linear layer of
    ``embedding_dim`` units. The score of a (user, item) pair is the raw dot
    product of the two embeddings (no normalisation), plus, if ``item_bias``
    is true, a **bias** that depends on the item: a single linear unit applied
    to the item features. The network scores a *positive* and a *negative*
    item for the same user, using **the same item tower and the same bias
    unit for both** (this weight sharing is what makes it siamese), and
    returns the difference of the two scores.

    Parameters
    ----------
    n_user_features : int
        Number of user features.
    n_item_features : int
        Number of item features.
    embedding_dim : int, optional
        Size of the embeddings.
    hidden : tuple of int, optional
        Sizes of the hidden layers of each tower.
    item_bias : bool, optional
        Add the item-dependent bias to the score.

    Returns
    -------
    model : keras.Model
        Takes the list ``[user_features, positive_item_features,
        negative_item_features]`` and returns ``score(u, positive) -
        score(u, negative)``, shape ``(batch, 1)``.
    user_tower : keras.Model
        Maps user features to embeddings. Shares its weights with ``model``.
    item_tower : keras.Model
        Maps item features to embeddings. Shares its weights with ``model``.
    bias_model : keras.Model or None
        Maps item features to the item bias, shape ``(batch, 1)``, sharing its
        weights with ``model``; ``None`` if ``item_bias`` is false.

    Notes
    -----
    Keras 3: do not apply TensorFlow functions (``tf.*``) to symbolic Keras
    tensors.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
model_t, ut_t, it_t, bm_t = build_siamese_bpr(21, 22, embedding_dim=8, hidden=(16,))
rng = np.random.default_rng(0)
xu, xp, xn = rng.normal(size=(6, 21)), rng.normal(size=(6, 22)), rng.normal(size=(6, 22))
# make the towers non-trivial, then compare the model with a score computed by hand from the pieces
for layer in (it_t.layers[-1], bm_t.layers[-1]):
    layer.set_weights([rng.normal(size=w.shape) for w in layer.get_weights()])
out = model_t.predict([xu, xp, xn], verbose=0)
score = lambda x: (ut_t.predict(xu, verbose=0) * it_t.predict(x, verbose=0)).sum(1) + bm_t.predict(x, verbose=0).ravel()
print("inputs:", len(model_t.inputs), "(expected 3) | output shape:", out.shape, "(expected (6, 1))")
print("model output = score(pos) - score(neg), computed from the towers:", bool(np.allclose(out.ravel(), score(xp) - score(xn), atol=1e-4)), "(expected True)")
no_bias = build_siamese_bpr(21, 22, embedding_dim=8, hidden=(16,), item_bias=False)
print("without item bias, bias_model is None:", no_bias[3] is None, "(expected True)")


## 5. Exercise 4 — the training loop

Keras `fit` would reuse the same triplets at every epoch. Write the loop yourself: fresh triplets at the start of each epoch, one epoch of `fit` on them. The target is irrelevant (the loss only looks at the score difference): pass zeros.

In [ ]:
def train_bpr(
    model,
    user_x: np.ndarray,
    item_x: np.ndarray,
    train: pd.DataFrame,
    user_ids: np.ndarray,
    item_ids: np.ndarray,
    epochs: int = 20,
    batch_size: int = 512,
    seed: int = 1,
) -> list[float]:
    """Train a siamese BPR model with fresh triplets at every epoch.

    At the start of each epoch, new triplets are drawn with
    ``sample_triplets`` (so that the negatives change from one epoch to the
    next), then the model is fitted for one epoch on them.

    Parameters
    ----------
    model : keras.Model
        Compiled model from ``build_siamese_bpr``, taking
        ``[user, positive item, negative item]`` features. Its targets are
        irrelevant: pass zeros.
    user_x : numpy.ndarray, shape (n_users, n_user_features)
        Standardised user features, row ``j`` for ``user_ids[j]``.
    item_x : numpy.ndarray, shape (n_items, n_item_features)
        Standardised item features, row ``i`` for ``item_ids[i]``.
    train : pandas.DataFrame
        Training ratings.
    user_ids, item_ids : numpy.ndarray
        Sorted ids of the users and of the catalogue.
    epochs : int, optional
        Number of epochs.
    batch_size : int, optional
        Mini-batch size.
    seed : int, optional
        Seed of the random generator used for the triplets.

    Returns
    -------
    list of float
        The training loss after each epoch.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
item_ids_t, item_raw_t = build_item_features(train_t, movies_t)
user_ids_t, user_raw_t = build_user_features(train_t, movies_t)
keras.utils.set_random_seed(0)
m_toy, _, _, _ = build_siamese_bpr(user_raw_t.shape[1], item_raw_t.shape[1], embedding_dim=4, hidden=(8,))
m_toy.compile(optimizer=keras.optimizers.Adam(1e-2), loss=bpr_loss)
hist = train_bpr(m_toy, user_raw_t, item_raw_t, train_t, user_ids_t, item_ids_t, epochs=3, batch_size=2)
print("epochs run:", len(hist), "(expected 3) | losses finite:", bool(np.isfinite(hist).all()))


## 6. Train it, and look at what it learned

`SiameseBPRRecommender` (provided) wires your functions together. Train it, then look at the loss and at the similar items given by the item tower (Session 05's `similar_items`, here with the dot-product space).

In [ ]:
class SiameseBPRRecommender(ScoreRecommender):
    """Recommender built on your siamese BPR network.

    Uses the feature functions of Session 05 and, from this session,
    ``build_siamese_bpr`` and ``train_bpr``.

    Parameters
    ----------
    embedding_dim : int, optional
        Size of the embeddings.
    hidden : tuple of int, optional
        Hidden layer sizes of each tower.
    epochs : int, optional
        Training epochs.
    batch_size : int, optional
        Mini-batch size.
    learning_rate : float, optional
        Adam learning rate.
    item_bias : bool, optional
        Use the item-dependent bias.
    seed : int, optional
        Seed for the weights and the triplet sampling.

    Attributes
    ----------
    model_, user_tower_, item_tower_, bias_model_ : keras.Model
        The trained networks (``bias_model_`` is None without item bias).
    user_emb_, item_emb_ : numpy.ndarray
        Embeddings of every user and catalogue item.
    item_bias_ : numpy.ndarray
        Bias of every catalogue item (zeros without item bias).
    history_ : list of float
        Training loss per epoch.
    """

    def __init__(self, embedding_dim: int = 32, hidden: tuple = (128, 64), epochs: int = 20, batch_size: int = 512,
                 learning_rate: float = 1e-3, item_bias: bool = True, seed: int = 1):
        self.embedding_dim, self.hidden, self.epochs, self.batch_size = embedding_dim, hidden, epochs, batch_size
        self.learning_rate, self.item_bias, self.seed = learning_rate, item_bias, seed

    def fit(self, train: pd.DataFrame, movies: pd.DataFrame) -> "SiameseBPRRecommender":
        """Build the features, train the siamese network and store the embeddings.

        Parameters
        ----------
        train : pandas.DataFrame
            Training ratings.
        movies : pandas.DataFrame
            Movie metadata.

        Returns
        -------
        SiameseBPRRecommender
            The fitted recommender.
        """
        self._prepare(train)
        item_ids, item_raw = build_item_features(train, movies)
        user_ids, user_raw = build_user_features(train, movies)
        assert np.array_equal(item_ids, self.items_), "item features must follow the sorted catalogue"
        item_x = (item_raw - item_raw.mean(0)) / (item_raw.std(0) + 1e-9)
        user_x = (user_raw - user_raw.mean(0)) / (user_raw.std(0) + 1e-9)
        self.user_pos_ = {int(u): j for j, u in enumerate(user_ids)}
        keras.utils.set_random_seed(self.seed)
        self.model_, self.user_tower_, self.item_tower_, self.bias_model_ = build_siamese_bpr(
            user_x.shape[1], item_x.shape[1], self.embedding_dim, self.hidden, self.item_bias)
        self.model_.compile(optimizer=keras.optimizers.Adam(self.learning_rate), loss=bpr_loss)
        self.history_ = train_bpr(self.model_, user_x, item_x, train, user_ids, item_ids,
                                  self.epochs, self.batch_size, self.seed)
        self.user_emb_ = self.user_tower_.predict(user_x, verbose=0)
        self.item_emb_ = self.item_tower_.predict(item_x, verbose=0)
        self.item_bias_ = (self.bias_model_.predict(item_x, verbose=0).ravel() if self.bias_model_ is not None
                           else np.zeros(len(item_ids)))
        return self

    def scores(self, user_id: int) -> np.ndarray:
        """Dot product with every item embedding, plus the item bias.

        Parameters
        ----------
        user_id : int
            User id. A user unseen in training gets the item biases only.

        Returns
        -------
        numpy.ndarray, shape (len(items_),)
            One score per catalogue item.
        """
        j = self.user_pos_.get(user_id)
        return self.item_bias_.copy() if j is None else self.item_emb_ @ self.user_emb_[j] + self.item_bias_


In [ ]:
siamese = SiameseBPRRecommender(epochs=20).fit(train, movies)      # about 30 seconds
print("loss per epoch:", [round(l, 3) for l in siamese.history_][::4], "...", round(siamese.history_[-1], 3))


## 7. Exercise 5 — is the difference real?

The leaderboard below ends with a decision: is the best model *really* better than the next one? Over the same 590 users, a difference of means is far less noisy than two separate standard errors suggest, because the users that are easy for one model are usually easy for the other. Implement `paired_difference`.

In [ ]:
def paired_difference(per_user_a: pd.Series, per_user_b: pd.Series) -> tuple[float, float]:
    """Mean difference between two models over the same users, with its standard error.

    Parameters
    ----------
    per_user_a, per_user_b : pandas.Series
        A metric (for example precision@10) of model A and of model B, indexed
        by user id. They cover the same users.

    Returns
    -------
    mean : float
        Mean over users of ``a - b``.
    standard_error : float
        ``std(a - b, ddof=1) / sqrt(n)``, the standard error of that mean.

    Notes
    -----
    TODO: implement this function.
    """
    raise NotImplementedError


In [ ]:
a = pd.Series([1.0, 2.0, 3.0], index=[1, 2, 3]); b = pd.Series([0.0, 2.0, 2.0], index=[1, 2, 3])
m, se = paired_difference(a, b)
print(f"mean {m:.4f} (expected 0.6667) | standard error {se:.4f} (expected 0.3333)")


## 8. The leaderboard

Every model of the course, on the same split, with the accuracy metrics of Session 04 **and** the beyond-accuracy ones. (About two minutes: it refits the latent-factor model and the two-tower network.)

In [ ]:
item_ids_all = np.array(sorted(train["movieId"].unique()))
item_vectors, item_index = build_genre_vectors(movies, item_ids_all)
item_counts = train.groupby("movieId").size().to_dict()
n_train_users = train["userId"].nunique()

models = {
    "Popularity (S01)": PopularityScorer().fit(train, movies),
    "Genre content-based (S02)": GenreContentScorer().fit(train, movies),
    "Latent factors (S03)": LatentFactorScorer().fit(train, movies),
    "Two-tower, retrieval (S05)": TwoTowerRecommender(objective="retrieval", epochs=30).fit(train, movies),
    "Siamese BPR (S06)": siamese,
}
per_user, rows = {}, {}
for name, model in models.items():
    per_user[name], rec_lists = evaluate_full(model, test, item_vectors, item_index, item_counts, n_train_users)
    rows[name] = summarise_full(per_user[name], rec_lists, n_catalogue=len(item_ids_all))
leaderboard = pd.DataFrame(rows).T
leaderboard.round(4)


Is the best model really better than popularity, and than the best earlier neural model? (paired, over the same users)

In [ ]:
best = leaderboard["precision"].idxmax()
print("best precision@10:", best)
for other in ["Popularity (S01)", "Two-tower, retrieval (S05)"]:
    if other != best:
        m, se = paired_difference(per_user[best]["precision"], per_user[other]["precision"])
        print(f"  {best} minus {other}: {m:+.4f} +/- {se:.4f}  ({m / se:.1f} standard errors)")


## 9. Log to the comparison table

In [ ]:
def log_results(comparison_path: str, results: pd.DataFrame, session: str = "S05") -> pd.DataFrame:
    """Append one row per model to the running comparison table.

    Parameters
    ----------
    comparison_path : str
        CSV path of the running table (created if missing).
    results : pandas.DataFrame
        Indexed by model name.
    session : str, optional
        Label written in a ``session`` column.

    Returns
    -------
    pandas.DataFrame
        The updated table.
    """
    new_rows = results.reset_index().rename(columns={"index": "model"})
    new_rows.insert(0, "session", session)
    if os.path.exists(comparison_path):
        new_rows = pd.concat([pd.read_csv(comparison_path), new_rows], ignore_index=True)
    new_rows.to_csv(comparison_path, index=False)
    return new_rows


In [ ]:
log_results("./comparison_table.csv", leaderboard[["precision", "precision_se", "ndcg", "coverage", "diversity", "novelty"]], session="S06").tail(len(leaderboard))


## 10. Closing: which model would you ship?

Write a short recommendation (half a page, in a markdown cell below) for **one** of these two products, using the leaderboard and the paired differences as evidence:

**A. A "top picks for you" shelf** on the home page of a streaming service. **B. A "discover something new" page** for users who feel they have seen everything.

Your text must use **at least three different metrics**, say which differences you consider real and which you do not (and why), and name one thing the leaderboard **cannot** tell you (and how you would find out).

## 11. Going further (optional)

**1. Cosine or dot?** Rebuild the network with L2-normalised embeddings and a fixed temperature. What happens to precision@10, and to the novelty of the lists? Why?

**2. Hard negatives.** Draw the negatives in proportion to item popularity instead of uniformly. What changes, and why is it a double-edged sword?

**3. A hybrid at the end.** Mix the siamese network with popularity as in Session 05. Does the hybrid still gain, now that the network ranks directly?

**4. Cold-start users.** Truncate the history of some users and evaluate the siamese network on them, with the protocol of the hybrid lecture (candidates exclude the user's real earlier ratings).

---
**End of the course. Thank you.**
